In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
from datetime import datetime
from connector import get_db_connection

In [2]:
conn = get_db_connection()
query = """
SELECT date_taken, date_taken_granularity, owner_nsid, owner_name
FROM photo 
WHERE date_taken IS NOT NULL
"""
df = pd.read_sql_query(query, conn)
conn.close()


/tmp/ipykernel_162252/1759468753.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


In [3]:
df['year'] = pd.to_datetime(df['date_taken'], errors='coerce').dt.year
df = df.dropna(subset=['year'])
df = df[(df['year'] >= 1850) & (df['year'] <= 2026)]
df = df[df['owner_nsid'] != '12403504@N02'] # filtering the british library because all their dates are 2013 (unreliable dates)
print(f"Loaded {len(df)} photos spanning {df['year'].min()}–{df['year'].max()}")

Loaded 467843 photos spanning 1850.0–2026.0


In [4]:
def plot_year_and_granularity(df):
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Photo Count per Year', 'Granularity (Mean ± Std per Year)'),
        vertical_spacing=0.2,
        row_heights=[0.7, 0.3]
    )
    year_counts = df['year'].value_counts().sort_index()
    fig.add_trace(
        go.Bar(
            x=year_counts.index,
            y=year_counts.values,
            name='Photo Count',
            marker_color='steelblue',
            opacity=0.8
        ),
        row=1, col=1
    )
    year_stats = df.groupby('year')['date_taken_granularity'].agg(['mean', 'std', 'count']).reset_index()
    year_stats['error'] = year_stats['std'] / np.sqrt(year_stats['count'])  # Standard error

    fig.add_trace(
        go.Bar(
            x=year_stats['year'],
            y=year_stats['mean'],
            name='Granularity Mean',
            marker_color='darkorange',
            opacity=0.8,
            error_y=dict(
                type='data',
                array=year_stats['error'],
                visible=True,
                color='black',
                thickness=1,
                width=3
            )
        ),
        row=2, col=1
    )

    fig.update_layout(
        height=700,
        showlegend=False,
        title_text="Photo Date Distribution & Granularity Analysis",
        title_font_size=20,
        xaxis_title="Year",
        yaxis_title="Number of Photos",
        xaxis2_title="Year",
        yaxis2_title="Granularity (Mean ± SEM)"
    )
    fig.update_xaxes(
        tickmode='linear',
        tick0=df['year'].min() // 10 * 10,
        dtick=10,
        row=1, col=1
    )
    fig.update_xaxes(
        tickmode='linear',
        tick0=df['year'].min() // 10 * 10,
        dtick=10,
        row=2, col=1
    )
    fig.show()
    print("\nSummary:")
    print(f"Total photos: {len(df)}")
    print(f"Years covered: {df['year'].min()}–{df['year'].max()}")
    print(f"Peak year: {year_counts.idxmax()} ({year_counts.max()} photos)")
    print(f"Avg photos/year: {len(df)/len(year_counts):.1f}")
    print(f"Avg granularity: {df['date_taken_granularity'].mean():.2f} ± {df['date_taken_granularity'].std():.2f}")

In [ ]:
def plot_granularity(df):
    year_stats = df.groupby('year')['date_taken_granularity'].agg(['mean', 'std', 'count']).reset_index()
    year_stats['error'] = year_stats['std'] / np.sqrt(year_stats['count'])  # Standard error
    
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=year_stats['year'],
            y=year_stats['mean'],
            name='Granularity Mean',
            marker_color='darkorange',
            opacity=0.8,
            error_y=dict(
                type='data',
                array=year_stats['error'],
                visible=True,
                color='black',
                thickness=1,
                width=3
            )
        )
    )
    
    fig.update_layout(
        xaxis_title="Year",
        yaxis_title="Granularity (Mean ± SEM)",
        title="Dating granularity by Year (8 = circa, 6 = year, 4 = month, 0 = second)",
        xaxis_tickangle=45
    )
    
    fig.update_xaxes(
        tickmode='linear',
        tick0=df['year'].min() // 10 * 10,
        dtick=10
    )
    
    fig.show()


In [6]:
def plot_year_per_institution(df, log_scale=True):
    """Simpler version using Plotly Express."""
    pivot_df = df.groupby(['year', 'owner_name']).size().reset_index(name='count')
    owner_totals = pivot_df.groupby('owner_name')['count'].sum().sort_values(ascending=False)
    owner_order = owner_totals.index.tolist()
    
    fig = px.bar(
        pivot_df, 
        x='year', 
        y='count', 
        color='owner_name',
        title='Pictures by Year by Institutions',
        barmode='stack',
        category_orders={'owner_name': owner_order},
        color_discrete_sequence=px.colors.qualitative.Light24 + px.colors.qualitative.Dark24 + 
                               px.colors.qualitative.Set3 + px.colors.qualitative.Pastel1 + 
                               px.colors.qualitative.Pastel2
    )
    
    fig.update_layout(
        xaxis_title='Year',
        yaxis_title='Count' + (' (Log Scale)' if log_scale else ''),
        legend_title='Owner Name',
        xaxis_tickangle=45,
        plot_bgcolor='white',  
    )
    fig.update_xaxes(
        tickmode='linear',
        tick0=df['year'].min() // 10 * 10,
        dtick=10,
    )
    fig.update_yaxes(
        gridcolor='lightgray' 
    ) 
    if log_scale:
        fig.update_layout(yaxis_type='log')
    
    fig.show()

In [7]:
plot_year_per_institution(df)
plot_granularity(df)

In [8]:
def _sample_uniform_low_granularity(year_group, max_per_year):
    sampled = []
    remaining = max_per_year
    sorted_group = year_group.sort_values('date_taken_granularity')
    gran_levels = sorted(sorted_group['date_taken_granularity'].unique())
    for gran in gran_levels:
        level_group = sorted_group[sorted_group['date_taken_granularity'] == gran]
        if len(sampled) + len(level_group) <= remaining:
            sampled.append(level_group)
            remaining -= len(level_group)
            if remaining <= 0:
                print(f"Error: sampled too much remaining: {remaining}")
                break
        else:
            critical_sample = _sample_uniform_owners(level_group, remaining)
            sampled.append(critical_sample)
            remaining = 0
            break
        
    return pd.concat(sampled, ignore_index=True)

def _sample_uniform_owners(subgroup, target_count):
    owner_sizes = subgroup.groupby('owner_nsid').size().sort_values()
    total_owners = len(owner_sizes)
    sampled = []
    remaining_target = target_count
    for i, (owner_nsid, size) in enumerate(owner_sizes.items()):
        remaining_owners = total_owners - i
        fair_share = remaining_target // remaining_owners
        take = min(size, fair_share)
        owner_photos = subgroup[subgroup['owner_nsid'] == owner_nsid]
        sampled.append(owner_photos.sample(n=take))
        remaining_target -= take 
    return pd.concat(sampled, ignore_index=True)

def sample_by_year(df, max_per_year=500):
    """
    Sample max_per_year photos for each year prioritizing:
    1. Lowest granularity first (take all until limit reached)
    2. Maximum uniform owner distribution in the critical granularity level
    """
    sampled_dfs = []   
    for year, group in df.groupby('year'):
        if len(group) > max_per_year:
            sampled = sampled = _sample_uniform_low_granularity(group, max_per_year)
            sampled_dfs.append(sampled)
        else:
            sampled_dfs.append(group)
    return pd.concat(sampled_dfs, ignore_index=True)



In [9]:
def get_cleaned_data_var_size(max_per_year=500):
    df_var_size = sample_by_year(df, max_per_year)
    print(f"Kept {len(df_var_size)} photos spanning {df_var_size['year'].min()}–{df_var_size['year'].max()}")
    plot_year_per_institution(df_var_size, log_scale=False)
    # plot_granularity(df300)
    return df_var_size

In [10]:
df3000 = get_cleaned_data_var_size(3000)
df1000 = get_cleaned_data_var_size(1000)
df700 = get_cleaned_data_var_size(700)
df500 = get_cleaned_data_var_size(500)

Kept 229109 photos spanning 1850.0–2026.0


Kept 134347 photos spanning 1850.0–2026.0


Kept 104534 photos spanning 1850.0–2026.0


Kept 79534 photos spanning 1850.0–2026.0


## TRAINING AI


In [11]:
# start with df500
# sample multiple datasets ?
# save training data in db ?
# encode images with SigLip (what size?) separate folder (encoder)
# save encoded vectors in new table (what dimensions ?)
# train regression model on vector space (what parameters ?)
